# v109_name_num_blocking — a name + address-number exact key on top of B6

| Field | Value |
|---|---|
| **Version** | `v109_name_num_blocking` |
| **Plan group** | A1 / A5 (blocking) |
| **Parent version** | v105 / v106 (B6 blocking) |
| **Author** | M1 rajaguru2004 |
| **Date** | 2026-09-26 |
| **Status** | blocking study (no matcher, no upload) |

Every earlier blocking number was measured on the 20 % val fold: 0.9906 pair recall at 33
candidates per S1. On test, each entity's top-k lists compete with 3–6× more same-name
records, so true records can fall off the lists. This study measures candidate recall on
the **mock fold** (test-shaped, `entity_resolution.mock`) for the current configuration and
for larger budgets, on a hashed sample of mock val entities blocked against their
country's **whole** mock pool (the density is the pool's, not the sample's).

Since v104, `candidate_pairs.tsv` holds only what survives the stage-1 filter (top 16 per
S1, p1 ≥ 0.01), so a larger blocking budget costs run time, not a larger candidate file.

## 1. Hypothesis

* **Change vs parent (B6):** one more exact pass, keyed on `name_core` plus one of the
  record's first four address numbers (`blocking.name_num_pass`, pool groups ≤ 100); the
  sim-first cap ranks its pairs first.
* **Why:** what B6 still misses is mostly "name close" (India 1,657 of 2,157 sampled misses):
  common names whose `name_core` group exceeds 200 records, so the exact pass skips them,
  and whose address shares no rare token with the S1. Name + house number keeps the group
  small.
* **Adopt if:** candidate recall gains ≥ 0.002 over B6 on the sampled mock val entities.

## 2. Setup

The configurations compared; everything else (normalisation, token map: v101's) is fixed.

In [1]:
import json
import time
from dataclasses import asdict, replace

import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.blocking import PASS_BITS, BlockingConfig, TopKSpec, block_partition
from entity_resolution.data import isin
from entity_resolution.evaluate import blocking_report, pair_in
from entity_resolution.features import FEATURE_COLUMNS, build_features
from entity_resolution.mock import build_mock, target_shape
from entity_resolution.pipeline import Fitted, PipelineConfig, load_normalised, pool_of
from entity_resolution.split import Fold, load_fold
from entity_resolution.tracking import log_result, timed
from entity_resolution.trainset import inner_split, sample_s1

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

EXP_DIR = C.EXPERIMENTS / "v109_name_num_blocking"
cfg = PipelineConfig()
token_map = json.loads((C.EXPERIMENTS / "v101_name_frequency" / "artifacts" /
                        "token_map.json").read_text())
B0 = cfg.blocking
ADDR_WORD = TopKSpec("addr_norm", "word", (1, 2), top_k=10, min_sim=0.3, max_df=0.01,
                     max_df_abs=10_000)


def variant(word_k=25, groups=50, addr=False, cap=60, order="exact_first") -> BlockingConfig:
    """A blocking budget: word top-k, exact group limit, address-word pass, cap, cap order."""
    return replace(B0, name_addr_word=replace(B0.name_addr_word, top_k=word_k),
                   exact_max_group=groups, addr_char=ADDR_WORD if addr else None,
                   max_per_s1=cap, cap_order=order)


# every configuration is derived from ONE uncapped superset blocking per country (word top-50,
# exact groups <= 200, address-word pass), by dropping pass bits and re-applying the cap
SUPER = replace(variant(word_k=50, groups=200, addr=True, cap=100_000, order="sim_first"),
                name_num_max_group=100)
B6 = variant(word_k=50, groups=200, addr=True, cap=100, order="sim_first")
CONFIGS = {
    "B6 (v106)": B6,
    "B6 + name+number <=100": replace(B6, name_num_max_group=100),
    "B6 + name+number, cap 120": replace(B6, name_num_max_group=100, max_per_s1=120),
}
N_SAMPLE = 20_000          # mock val entities per country
timings: dict[str, float] = {}
t_start = time.time()
{k: v.key() for k, v in CONFIGS.items()} | {"SUPER": SUPER.key()}

{'B6 (v106)': 'b54f2a3d',
 'B6 + name+number <=100': '3542fba2',
 'B6 + name+number, cap 120': 'db1f33c4',
 'SUPER': '4d61ab8d'}

## 3. Data

The mock fold of v103/v104, then a hashed sample of its val entities per country (seed 7,
`sample_s1`), each blocked against the country's whole mock pool.

In [2]:
with timed("load", timings):
    train = load_fold("train", columns=[C.COUNTRY])
    val = load_fold("val", columns=[C.COUNTRY])
    fit_fold, tune_fold = inner_split(train)
    fit_sample = sample_s1(fit_fold.s1, cfg.n_fit_s1)[C.ENTITY_ID]
    mock = build_mock(train, val, tune_fold.s1[C.ENTITY_ID], fit_sample, target_shape())
del train, val, fit_fold, tune_fold, fit_sample
part = mock.part("val")
samples = {}
for country in sorted(part.s1[C.COUNTRY].unique()):
    s1c = part.s1[(part.s1[C.COUNTRY] == country).to_numpy()]
    samples[country] = sample_s1(s1c, N_SAMPLE)
{c: len(s) for c, s in samples.items()}

{'India': 19937, 'US': 20364}

## 4. Method

One **superset** blocking per country (word top-50, exact groups ≤ 200, address-word pass,
no cap) of the sampled entities against the country's whole mock pool; every configuration
is then derived from it (`derive`): exact bits dropped where the pool record's key group
exceeds the limit, word bits beyond the configuration's top-k (rank by cosine inside the
S1), the address pass removed when disabled, and the cap re-applied in the configuration's
order. Recall, entity recall, the ceiling F0.5 of a perfect matcher and candidates per S1
come from `evaluate.blocking_report` on the sampled entities.

In [3]:
EXACT_KEYS = (("name_core", 1), ("name_sorted", 2), ("name_squash", 4))


def derive(sup: pd.DataFrame, gsize: dict, bcfg: BlockingConfig) -> pd.DataFrame:
    """The candidates ``bcfg`` would produce, from the uncapped superset ``sup``.

    Exact bits go where the pool record's key group exceeds the limit, word-pass bits beyond
    the word top-k (rank by cosine inside the S1), the address pass when disabled; pairs left
    without a bit leave; the cap is re-applied with the configuration's order.
    """
    bits = sup["pass"].to_numpy().astype(np.int64)
    for key, bit in EXACT_KEYS:
        bits[gsize[key] > bcfg.exact_max_group] &= ~bit
    word = (bits & 16) != 0
    w = pd.Series(np.where(word, sup["sim_name_addr_word"].to_numpy(), -np.inf))
    rank = w.groupby(sup[C.S1_ID].to_numpy()).rank(method="first", ascending=False).to_numpy()
    bits[word & (rank > bcfg.name_addr_word.top_k)] &= ~16
    if bcfg.addr_char is None:
        bits &= ~32
    if bcfg.name_num_max_group is None:
        bits &= ~64
    keep = bits != 0
    out = sup.loc[keep, [C.S1_ID, C.ENTITY_ID]].reset_index(drop=True)
    b = bits[keep]
    sims = np.stack([np.where((b & 8) != 0, sup["sim_name_char"].to_numpy()[keep], np.nan),
                     np.where((b & 16) != 0, sup["sim_name_addr_word"].to_numpy()[keep], np.nan),
                     np.where((b & 32) != 0, sup["sim_addr_char"].to_numpy()[keep], np.nan)], 1)
    best = np.nan_to_num(np.nanmax(np.where(np.isnan(sims), -np.inf, sims), axis=1),
                         neginf=np.nan)
    exact = (b & 7) != 0
    if bcfg.cap_order == "sim_first":           # name + number pairs first (blocking.py)
        key1, key2 = (~((b & 64) != 0)).astype(float), np.where(np.isnan(best), -1.0, best)
    else:
        key1, key2 = (~exact).astype(float), np.nan_to_num(best, nan=0.0)
    order = np.lexsort((out[C.ENTITY_ID].to_numpy(), -key2, key1, out[C.S1_ID].to_numpy()))
    ranked = out.iloc[order]
    r = ranked.groupby(C.S1_ID, sort=False).cumcount().to_numpy()
    return ranked[r < bcfg.max_per_s1].reset_index(drop=True)


rows, pairs_by = [], {}
for country, s1s in samples.items():
    s1n = load_normalised("train", (1,), cfg, s1s[C.ENTITY_ID], token_map)
    pooln = load_normalised("train", (2, 3), cfg, pool_of(mock.fold)[C.ENTITY_ID], token_map,
                            country=country)
    ids = pd.Index(s1s[C.ENTITY_ID])
    fold = Fold(f"sample_{country}", s1s.reset_index(drop=True), part.s2, part.s3,
                part.pairs[isin(part.pairs[C.S1_ID], ids)].reset_index(drop=True))
    t0 = time.time()
    sup = block_partition(s1n, pooln, SUPER)
    super_secs = time.time() - t0
    by_pool = pooln.set_index(C.ENTITY_ID)
    gsize = {}
    for key, _ in EXACT_KEYS:
        counts = pooln[key].value_counts()
        gsize[key] = sup[C.ENTITY_ID].map(by_pool[key]).map(counts).fillna(0).to_numpy()
    print(country, f"superset {super_secs:.0f}s, {len(sup) / len(s1s):.1f} pairs/S1", flush=True)
    for name, bcfg in CONFIGS.items():
        pairs = derive(sup, gsize, bcfg)
        rep = blocking_report(pairs, fold)
        rows.append({"country": country, "config": name, **rep})
        pairs_by[(country, name)] = pairs
        print(country, name, f"recall {rep['pair_recall']:.4f} cands {rep['candidates_mean']:.1f}",
              flush=True)
    timings[f"superset_{country}_seconds"] = round(super_secs, 1)
    del pooln, by_pool, sup
report = pd.DataFrame(rows)
report[["country", "config", "pair_recall", "entity_recall", "ceiling_f_beta",
        "candidates_mean", "candidates_p95"]]

India superset 194s, 94.7 pairs/S1


India B6 (v106) recall 0.9688 cands 67.8


India B6 + name+number <=100 recall 0.9776 cands 69.3


India B6 + name+number, cap 120 recall 0.9778 cands 73.6


US superset 147s, 68.5 pairs/S1


US B6 (v106) recall 0.9870 cands 63.3


US B6 + name+number <=100 recall 0.9878 cands 63.4


US B6 + name+number, cap 120 recall 0.9879 cands 65.1


,country,config,pair_recall,entity_recall,ceiling_f_beta,candidates_mean,candidates_p95
0,India,B6 (v106),0.968838,0.997247,0.989331,67.754677,100.0
1,India,B6 + name+number <=100,0.977636,0.998571,0.993013,69.337463,100.0
2,India,"B6 + name+number, cap 120",0.977838,0.998571,0.993082,73.584240,120.0
3,US,B6 (v106),0.987037,0.998856,0.995648,63.304704,100.0
4,US,B6 + name+number <=100,0.987767,0.999064,0.996018,63.356953,100.0
5,US,"B6 + name+number, cap 120",0.987867,0.999064,0.996043,65.116627,120.0


Both countries pooled (weighted by true pairs), and per-pass recall of the current and the
largest configuration.

In [4]:
def pooled(name: str) -> dict:
    """Recall and candidates of one configuration over both countries' samples."""
    tp = n_true = n_cand = n_s1 = 0
    for country, s1s in samples.items():
        pairs = pairs_by[(country, name)]
        truth = part.pairs[isin(part.pairs[C.S1_ID], pd.Index(s1s[C.ENTITY_ID]))]
        tp += int(pair_in(truth, pairs).sum())
        n_true += len(truth)
        n_cand += len(pairs)
        n_s1 += len(s1s)
    return {"pair_recall": tp / n_true, "cands_per_s1": n_cand / n_s1,
            "missed_pairs": n_true - tp}


overall = pd.DataFrame({name: pooled(name) for name in CONFIGS}).T
display(overall)
per_pass = []  # pass bits are not kept by derive(); see the superset timings instead
pd.DataFrame(per_pass)

,pair_recall,cands_per_s1,missed_pairs
B6 (v106),0.977981,65.506116,3063.0
B6 + name+number <=100,0.982726,66.315526,2403.0
"B6 + name+number, cap 120",0.982877,69.305576,2382.0


""


## 5. Evaluation

The decision table: recall gained against B0, and the cost as the ratio of candidates per
S1 (features and stage 1 scale with it; the blocking itself is ~10 min either way).

In [5]:
base = overall.loc["B6 (v106)"]
# run time grows with the candidates the blocking keeps (features and stage 1 dominate)
decision = overall.assign(recall_gain=overall["pair_recall"] - base["pair_recall"],
                          time_ratio=overall["cands_per_s1"] / base["cands_per_s1"])
decision

,pair_recall,cands_per_s1,missed_pairs,recall_gain,time_ratio
B6 (v106),0.977981,65.506116,3063.0,0.000000,1.000000
B6 + name+number <=100,0.982726,66.315526,2403.0,0.004744,1.012356
"B6 + name+number, cap 120",0.982877,69.305576,2382.0,0.004895,1.058002


## 6. Error analysis

What the highest-recall configuration still misses: both normalised records side by side with
their name and address similarity, to separate hopeless renames from fixable misses.

In [6]:
def misses(name: str, country: str, n: int = 15) -> pd.DataFrame:
    """Sampled true pairs missing from a configuration's candidates, with similarities."""
    s1s = samples[country]
    truth = part.pairs[isin(part.pairs[C.S1_ID], pd.Index(s1s[C.ENTITY_ID]))]
    miss = truth[~pair_in(truth, pairs_by[(country, name)])]
    miss = miss.sort_values([C.S1_ID, C.ENTITY_ID], ignore_index=True)  # S1 groups contiguous
    s1n = load_normalised("train", (1,), cfg, miss[C.S1_ID].drop_duplicates(), token_map)
    pooln = load_normalised("train", (2, 3), cfg, miss[C.ENTITY_ID], token_map)
    X = build_features(miss.assign(**{"pass": np.uint8(0), "sim_name_char": np.nan,
                                      "sim_name_addr_word": np.nan, "sim_addr_char": np.nan}),
                       s1n, pooln, groups=("name_fuzzy", "address"))
    left = s1n.set_index(C.ENTITY_ID)
    right = pooln.set_index(C.ENTITY_ID)
    show = pd.DataFrame({"name_norm_l": miss[C.S1_ID].map(left["name_norm"]),
                         "addr_norm_l": miss[C.S1_ID].map(left["addr_norm"]),
                         "name_norm_r": miss[C.ENTITY_ID].map(right["name_norm"]),
                         "addr_norm_r": miss[C.ENTITY_ID].map(right["addr_norm"])})
    X = X.reset_index(drop=True)
    stats = pd.DataFrame({"core_token_set": X["core_token_set"], "ad_token_set": X["ad_token_set"]})
    cats = pd.cut(stats["core_token_set"].fillna(0), [-0.01, 0.3, 0.7, 1.0],
                  labels=["name unrelated", "name partial", "name close"])
    print(country, name, len(miss), "missed pairs;", cats.value_counts().to_dict())
    return pd.concat([show[["name_norm_l", "addr_norm_l", "name_norm_r", "addr_norm_r"]],
                      stats], axis=1).head(n)


BEST = decision["pair_recall"].idxmax()
for country in samples:
    display(misses(BEST, country))

India B6 + name+number, cap 120 1534 missed pairs; {'name close': 1035, 'name partial': 394, 'name unrelated': 105}


,name_norm_l,addr_norm_l,name_norm_r,addr_norm_r,core_token_set,ad_token_set
0,ayur and sons corp,bodai panchayat jugberia gali no 3 sodepur ghola jugberi...,ayur + ss corp,no 280 bndai panchayat calcutta howrah wb,0.727273,0.861111
1,am services pvt ltd,s no 19 1 8 b hingane home col pune city pune mh,am pvt ltd center,,0.500000,NaN
2,am services pvt ltd,s no 19 1 8 b hingane home col pune city pune mh,mr am pvt ltd center,,0.500000,NaN
3,supreme global private limited,ist fl beri wala bagh azad mkt chowk dl n delhi 7116,supreme global private limited,dl dl 711 n delhi,1.000000,0.833333
4,supreme global private limited,ist fl beri wala bagh azad mkt chowk dl n delhi 7116,supreme glrrl private limited,711 dl n delhi dl,0.814815,0.833333
5,rural clinic private limited,c 120 basement stn plz stn rd bhandup we st mumbai mumba...,rcprivate,c 120 mumbai mumbai city mh,0.285714,1.000000
6,stalwart electronics corporation,h no 72 village ghondli dl dl n e dl,smt beloquo,72 dl n e dl,0.222222,1.000000
7,international green private limited,a 188 4 and a 141 3 fl abul fazal enclave 2 jamia nagar ...,international green psmrvae limited,,1.000000,NaN
8,aditya builders pvt ltd,45 hazra rd calcutta howrah wb,aditya pvt ltd center,howrah wb hazra rd calcutta c 45,0.642857,1.000000
9,ak institute,l s no 3298 rajpur kadi mahesana gj,ak insttiute,,0.916667,NaN


US B6 + name+number, cap 120 848 missed pairs; {'name close': 594, 'name partial': 209, 'name unrelated': 45}


,name_norm_l,addr_norm_l,name_norm_r,addr_norm_r,core_token_set,ad_token_set
0,superior first asset,2137 st louis ave chicago il,superior asset first id 24995,il chicago 137 st louis ave,1.000000,0.981818
1,claybrooks controls inc,2143 chaffe ct owensboro ky,qu0drex,2143 chaffe ct ownsboro cdp ky,0.153846,0.912281
2,youth coalition,51 franklin rd newport news city va,youth services,52 franklin rd newpor news va,0.526316,0.875000
3,vh frequency pc,563 brigich rd chartiers township pa,vh frerqurency pc,563 brigich rd canonsburg pa,0.923077,0.755556
4,harbor assets llc,408 stroll ave duluth mn,onyxpyra,mn 408 stroll ave duuth,0.190476,0.978723
5,a select mines,816 dogwood dr berea ky,a select,814 dogwood dr madison county ky,1.000000,0.722222
6,valley preparatory academy inc,5501 5 st wa dc,valley academy inc service service,5501 5 st washngton dc,0.777778,0.888889
7,family clinic llc,in 859 pike st martinsville,family clinic llc enterprises,,1.000000,NaN
8,delta outdoor,110 hampshire rd wellesley ma,delta service,,0.555556,NaN
9,zaex llc,1106 venable rd fluvanna county va,zaex llc services,1106 venable rgad palmyra va,1.000000,0.697674


## 7. Log the result

A blocking study: `cand_recall` is the pooled mock-val-sample recall of the adopted
configuration; F0.5 columns stay empty (no matcher here).

In [7]:
gain_ok = decision[(decision["recall_gain"] >= 0.002) & (decision["time_ratio"] <= 1.5)]
ADOPT = (gain_ok.sort_values("pair_recall", ascending=False).index[0] if len(gain_ok)
         else "B6 (v106)")
record = {
    "hypothesis": "candidate recall drops at test density; larger word top-k and caps buy it back",
    "configs": {k: asdict(v) for k, v in CONFIGS.items()}, "superset": asdict(SUPER),
    "config_keys": {k: v.key() for k, v in CONFIGS.items()} | {"SUPER": SUPER.key()},
    "report": report.to_dict("records"), "overall": overall.to_dict("index"),
    "decision": decision.to_dict("index"), "adopted": ADOPT, **timings,
}
print("adopt:", ADOPT)
row = log_result(
    EXP_DIR, change="name + address-number exact key on top of B6 (sampled mock val entities)",
    group="A1", cand_recall=float(overall.loc[ADOPT, "pair_recall"]),
    notes=(f"B6 recall {overall.loc['B6 (v106)', 'pair_recall']:.4f}; adopted {ADOPT}: "
           f"{overall.loc[ADOPT, 'pair_recall']:.4f} at {overall.loc[ADOPT, 'cands_per_s1']:.1f}/S1"),
    metrics=record, owner="M1", parent="v105", decision="KEEP" if ADOPT != "B6 (v106)" else "DROP")
print(f"notebook total {time.time() - t_start:.0f} s")
row

adopt: B6 + name+number, cap 120
notebook total 506 s


{'version': 'v109',
 'date': '2026-09-26',
 'group': 'A1',
 'change': 'name + address-number exact key on top of B6 (sampled mock val entities)',
 'local_f05': '',
 'mock_f05': '',
 'cand_recall': '0.9829',
 'public_f05': '',
 'commit': 'a241e40',
 'notes': 'B6 recall 0.9780; adopted B6 + name+number, cap 120: 0.9829 at 69.3/S1',
 'owner': 'M1',
 'parent': 'v105',
 'decision': 'KEEP'}

## 8. Conclusion

Written after the run from the numbers above.